# **Space X Falcon 9 First Stage Landing Prediction**

## Web Scraping Falcon 9 and Falcon Heavy Launch Records from Wikipedia

Estimated time needed: **40** minutes

In this lab, we will perform web scraping to collect Falcon 9 historical launch records
from a Wikipedia page titled `List of Falcon 9 and Falcon Heavy launches`.

https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches

### Objectives

- Extract a Falcon 9 launch records HTML table from Wikipedia
- Parse the table and convert it into a Pandas DataFrame
- Save the scraped data to CSV

---
## Import Libraries

In [1]:
import sys
import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd
import os

print('Libraries loaded.')

Libraries loaded.


## Define Helper Functions

These helper functions parse individual table cells from the Wikipedia HTML.

In [2]:
def date_time(table_cells):
    """Return the date and time from an HTML table cell."""
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]


def booster_version(table_cells):
    """Return the booster version from an HTML table cell."""
    out = ''.join([booster_version for i, booster_version
                   in enumerate(table_cells.strings) if i % 2 == 0][0:-1])
    return out


def landing_status(table_cells):
    """Return the landing status from an HTML table cell."""
    out = [i for i in table_cells.strings][0]
    return out


def get_mass(table_cells):
    """Extract payload mass string from an HTML table cell."""
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass.find("kg")
        new_mass = mass[0:mass.find("kg") + 2]
    else:
        new_mass = 0
    return new_mass


def extract_column_from_header(row):
    """Extract column name from a table header element."""
    if row.br:
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()

    column_name = ' '.join(row.contents)

    # Filter out digit-only and empty names
    if not (column_name.strip().isdigit()):
        column_name = column_name.strip()
        return column_name

print('Helper functions defined.')

Helper functions defined.


## TASK 1: Request the Falcon 9 Launch Wiki Page

We use a snapshot of the Wikipedia page updated on **9th June 2021** for consistency.

In [3]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/91.0.4472.124 Safari/537.36"
}

# Perform the HTTP GET request
response = requests.get(static_url, headers=headers, timeout=30)
print(f'Status code: {response.status_code}')

Status code: 200


In [4]:
# Create a BeautifulSoup object from the response
soup = BeautifulSoup(response.text, 'html.parser')
print(f'Page title: {soup.title.string}')

Page title: List of Falcon 9 and Falcon Heavy launches - Wikipedia


## TASK 2: Extract Column Names from the HTML Table Header

Find all tables, then extract `<th>` elements from the launch table to get column names.

In [5]:
# Find all tables on the wiki page
html_tables = soup.find_all('table')
print(f'Total tables found: {len(html_tables)}')

Total tables found: 25


In [6]:
# The third table contains the launch records
first_launch_table = html_tables[2]

# Extract column names from <th> elements
column_names = []

for th in first_launch_table.find_all('th'):
    name = extract_column_from_header(th)
    if name is not None and len(name) > 0:
        column_names.append(name)

print(f'Extracted columns: {column_names}')

Extracted columns: ['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


## TASK 3: Create a DataFrame by Parsing the Launch HTML Tables

Initialize a dictionary with the column names and populate it by iterating through the table rows.

In [7]:
launch_dict = dict.fromkeys(column_names)

# Remove the irrelevant 'Date and time ( )' column
del launch_dict['Date and time ( )']

# Initialize each value as an empty list
launch_dict['Flight No.'] = []
launch_dict['Launch site'] = []
launch_dict['Payload'] = []
launch_dict['Payload mass'] = []
launch_dict['Orbit'] = []
launch_dict['Customer'] = []
launch_dict['Launch outcome'] = []
# Added columns
launch_dict['Version Booster'] = []
launch_dict['Booster landing'] = []
launch_dict['Date'] = []
launch_dict['Time'] = []

print(f'Dictionary keys: {list(launch_dict.keys())}')

Dictionary keys: ['Flight No.', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome', 'Version Booster', 'Booster landing', 'Date', 'Time']


In [8]:
extracted_row = 0

# Extract each table
for table_number, table in enumerate(soup.find_all('table', "wikitable plainrowheaders collapsible")):
    # Get table rows
    for rows in table.find_all("tr"):
        # Check if first table heading is a number (flight number)
        if rows.th:
            if rows.th.string:
                flight_number = rows.th.string.strip()
                flag = flight_number.isdigit()
        else:
            flag = False

        # Get table data elements
        row = rows.find_all('td')

        # If it is a number, save cells into the dictionary
        if flag:
            extracted_row += 1

            # Flight Number
            launch_dict['Flight No.'].append(flight_number)

            # Date and Time
            datatimelist = date_time(row[0])
            date = datatimelist[0].strip(',')
            launch_dict['Date'].append(date)
            time = datatimelist[1]
            launch_dict['Time'].append(time)

            # Booster version
            bv = booster_version(row[1])
            if not bv:
                bv = row[1].a.string if row[1].a else None
            launch_dict['Version Booster'].append(bv)

            # Launch Site
            launch_site = row[2].a.string if row[2].a else None
            launch_dict['Launch site'].append(launch_site)

            # Payload
            payload = row[3].a.string if row[3].a else None
            launch_dict['Payload'].append(payload)

            # Payload Mass
            payload_mass = get_mass(row[4])
            launch_dict['Payload mass'].append(payload_mass)

            # Orbit
            orbit = row[5].a.string if row[5].a else None
            launch_dict['Orbit'].append(orbit)

            # Customer
            customer = row[6].a.string if row[6].a else row[6].text.strip()
            launch_dict['Customer'].append(customer)

            # Launch outcome
            launch_outcome = list(row[7].strings)[0]
            launch_dict['Launch outcome'].append(launch_outcome)

            # Booster landing
            booster_landing = landing_status(row[8])
            launch_dict['Booster landing'].append(booster_landing)

print(f'Total rows extracted: {extracted_row}')

Total rows extracted: 121


## Create the DataFrame and Inspect

In [9]:
df = pd.DataFrame({key: pd.Series(value) for key, value in launch_dict.items()})
print(f'DataFrame shape: {df.shape}')
df.head(10)

DataFrame shape: (121, 11)


,Flight No.,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Version Booster,Booster landing,Date,Time
0,1,CCAFS,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success\n,F9 v1.07B0003.18,Failure,4 June 2010,18:45
1,2,CCAFS,Dragon,0,LEO,NASA,Success,F9 v1.07B0004.18,Failure,8 December 2010,15:43
2,3,CCAFS,Dragon,525 kg,LEO,NASA,Success,F9 v1.07B0005.18,No attempt\n,22 May 2012,07:44
3,4,CCAFS,SpaceX CRS-1,"4,700 kg",LEO,NASA,Success\n,F9 v1.07B0006.18,No attempt,8 October 2012,00:35
4,5,CCAFS,SpaceX CRS-2,"4,877 kg",LEO,NASA,Success\n,F9 v1.07B0007.18,No attempt\n,1 March 2013,15:10
5,6,VAFB,CASSIOPE,500 kg,Polar orbit,MDA,Success,F9 v1.17B10038,Uncontrolled,29 September 2013,16:00
6,7,CCAFS,SES-8,"3,170 kg",GTO,SES,Success,F9 v1.1,No attempt,3 December 2013,22:41
7,8,CCAFS,Thaicom 6,"3,325 kg",GTO,Thaicom,Success,F9 v1.1,No attempt,6 January 2014,22:06
8,9,Cape Canaveral,SpaceX CRS-3,"2,296 kg",LEO,NASA,Success\n,F9 v1.1,Controlled,18 April 2014,19:25
9,10,Cape Canaveral,Orbcomm-OG2,"1,316 kg",LEO,Orbcomm,Success,F9 v1.1,Controlled,14 July 2014,15:15


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Flight No.       121 non-null    object
 1   Launch site      121 non-null    object
 2   Payload          121 non-null    object
 3   Payload mass     121 non-null    object
 4   Orbit            121 non-null    object
 5   Customer         120 non-null    object
 6   Launch outcome   121 non-null    object
 7   Version Booster  121 non-null    object
 8   Booster landing  121 non-null    object
 9   Date             121 non-null    object
 10  Time             121 non-null    object
dtypes: object(11)
memory usage: 10.5+ KB


## Save to CSV

In [11]:
os.makedirs('../data', exist_ok=True)
output_path = '../data/spacex_web_scraped.csv'
df.to_csv(output_path, index=False)
print(f'Saved {len(df)} rows to {output_path}')

Saved 121 rows to ../data/spacex_web_scraped.csv


## Summary

In this notebook we:

1. **Requested** the Falcon 9 launch records page from Wikipedia (static snapshot)
2. **Parsed** the HTML using BeautifulSoup to find launch tables
3. **Extracted** column headers from `<th>` elements
4. **Iterated** through table rows to extract each launch record
5. **Saved** the scraped dataset as `data/spacex_web_scraped.csv`

Columns extracted: Flight No., Date, Time, Version Booster, Launch site,
Payload, Payload mass, Orbit, Customer, Launch outcome, Booster landing